In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from imblearn.over_sampling import RandomOverSampler
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import string
import random

In [2]:
max_raw = 5000
data = pd.read_csv(
    r"C:\Users\asus2022\OneDrive - New Ismailia National University\Documents\drasa\GAM3A\Level3 eng\SECOND TERM\Optmize\Data\SMSSpamCollection.csv",
    sep="\t",
    header=None,
    nrows=max_raw,
    names=["label", "data_text"]
)

In [3]:
# Preprocess text data
ps = PorterStemmer()
stopwords_en = set(stopwords.words('english'))

def clean_text(text):
    # Remove punctuation and lowercase
    text = text.translate(str.maketrans('', '', string.punctuation)).lower()
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and stem
    cleaned_tokens = [ps.stem(word) for word in tokens if word not in stopwords_en]
    return ' '.join(cleaned_tokens)
# Apply cleaning
data['cleaned_text'] = data['data_text'].apply(clean_text)
X = data['cleaned_text']
y = data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [4]:
# Tokenization and sequencing
max_words = 5000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_padded = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')


In [5]:
# Encode labels
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Convert to numpy arrays
y_train_encoded = np.array(y_train_encoded, dtype=np.float32)
y_test_encoded = np.array(y_test_encoded, dtype=np.float32)


In [6]:
class_weight = {0: 1., 1: 10.}

In [7]:
def fitness_function(params):
    # Number of layers and neurons from the params
    num_layers = int(params[0])  # Number of layers (1 or 2)
    neurons = int(params[1])  # Number of neurons per layer
    
    # Build the model dynamically based on the number of layers and neurons
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
    
    for _ in range(num_layers):
        model.add(tf.keras.layers.LSTM(neurons, return_sequences=True))
    
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    
    # Compile the model
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Train the model for 1 epoch (for testing purposes) with class weights
    history = model.fit(
        X_train_padded, y_train_encoded,
        epochs=10,
        batch_size=64,
        validation_data=(X_test_padded, y_test_encoded),
        class_weight=class_weight,  # Pass the manually specified class weights here
        verbose=0
    )
    
    # Extract the validation accuracy from the training process
    val_acc = history.history['val_accuracy'][-1]
    
    # Return negative accuracy since the optimizer minimizes
    return -val_acc


In [8]:
def GWO(objective_function, l_boundry, u_boundry, dim, SearchAgents_no, Max_iter):
    Alpha_pos = np.zeros(dim)
    Alpha_score = float("inf")
    
    Beta_pos = np.zeros(dim)
    Beta_score = float("inf")
    
    Delta_pos = np.zeros(dim)
    Delta_score = float("inf")
    
    # Initialize positions of wolves randomly within the boundaries
    Positions = np.random.uniform(0, 1, (SearchAgents_no, dim)) * (np.array(u_boundry) - np.array(l_boundry)) + np.array(l_boundry)
    
    for l in range(Max_iter):
        for i in range(SearchAgents_no):
            # Apply bounds to positions
            Positions[i,:] = np.clip(Positions[i,:], l_boundry, u_boundry)
            
            # Evaluate fitness
            fitness = objective_function(Positions[i,:])
            
            # Update Alpha, Beta, Delta positions and scores
            if fitness < Alpha_score:
                Alpha_score = fitness
                Alpha_pos = Positions[i,:].copy()
            elif fitness < Beta_score:
                Beta_score = fitness
                Beta_pos = Positions[i,:].copy()
            elif fitness < Delta_score:
                Delta_score = fitness
                Delta_pos = Positions[i,:].copy()
        
        # Update the position of search agents
        a = 2 - l * (2 / Max_iter)
        for i in range(SearchAgents_no):
            for j in range(dim):
                r1 = random.random()
                r2 = random.random()
                
                # Update the positions based on the alpha, beta, and delta wolves
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * Alpha_pos[j] - Positions[i,j])
                X1 = Alpha_pos[j] - A1 * D_alpha
                
                r1 = random.random()
                r2 = random.random()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * Beta_pos[j] - Positions[i,j])
                X2 = Beta_pos[j] - A2 * D_beta
                
                r1 = random.random()
                r2 = random.random()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * Delta_pos[j] - Positions[i,j])
                X3 = Delta_pos[j] - A3 * D_delta
                
                # Average the positions to update the wolves' positions
                Positions[i,j] = (X1 + X2 + X3) / 3.0
    
    return Alpha_pos, Alpha_score

In [9]:
lb = [1, 32]  # Lower bounds: 1 layer and at least 32 neurons per layer
ub = [2, 128]  # Upper bounds: 2 layers and at most 128 neurons per layer
dim = 2  # 1 dimension for the number of layers + 1 dimension for neurons per layer
num_wolves = 5
max_iter = 10
# Perform optimization using the GWO function
best_params, best_score = GWO(fitness_function, lb, ub, dim, num_wolves, max_iter)

# Print the best parameters and best score
print(f"Number of layers: {int(round(best_params[0]))}")
print(f"Number of neurons: {int(round(best_params[1]))}")
print("Best Parameters:", best_params)
print("Best Score (Validation Accuracy):", -best_score)


C:\Users\asus2022\OneDrive - New Ismailia National University\Documents\Anaconda\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Number of layers: 1
Number of neurons: 121
Best Parameters: [  1.12588033 120.83896936]
Best Score (Validation Accuracy): 0.9909999966621399


In [10]:
# استخدم أفضل المعلمات التي تم العثور عليها
best_num_layers = int(round(best_params[0]))  # عدد الطبقات المثلى
best_neurons = int(round(best_params[1]))  # عدد النيورنز المثلى

# بناء الموديل باستخدام هذه المعلمات
model = tf.keras.Sequential()
model.add(tf.keras.layers.Embedding(input_dim=max_words, output_dim=128, input_length=max_len))

# إضافة الطبقات بناءً على المعلمات المثلى
for _ in range(best_num_layers):
    model.add(tf.keras.layers.LSTM(best_neurons, return_sequences=True))

model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(32, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

# تجميع الموديل
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# تدريب الموديل مرة أخرى لفترة أطول
history = model.fit(
    X_train_padded, y_train_encoded,
    epochs=10,  # عدد أكبر من العصور
    batch_size=64,
    validation_data=(X_test_padded, y_test_encoded),
    class_weight=class_weight,
    verbose=1
)
# model.save_weights('best_model_weights.h5')
# الآن يمكننا استخدام الموديل المدرب لإجراء التنبؤات
predictions = model.predict(X_test_padded)

# إذا أردت التنبؤ بالفئة
predicted_classes = (predictions > 0.5).astype("int32")  # تحويل التنبؤات إلى فئات (0 أو 1)


Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.6250 - loss: 1.0115 - val_accuracy: 0.9610 - val_loss: 0.2379
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.9838 - loss: 0.1697 - val_accuracy: 0.9820 - val_loss: 0.0469
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.9958 - loss: 0.0576 - val_accuracy: 0.9810 - val_loss: 0.0465
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.9949 - loss: 0.0351 - val_accuracy: 0.9830 - val_loss: 0.0629
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.9978 - loss: 0.0158 - val_accuracy: 0.9850 - val_loss: 0.0558
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 1.0000 - loss: 7.1358e-04 - val_accuracy: 0.9830 - val_loss: 0.0613
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 1.0000 - loss: 1.4354e-04 - val_accuracy: 0.9850 - val_loss: 0.0659
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - accuracy: 1.0000 - loss: 1.5057e-04 - val_accuracy

ValueError: The filename must end in `.weights.h5`. Received: filepath=best_model_weights.h5

In [11]:
model.save_weights('best_model.weights.h5')


In [13]:
model.load_weights('best_model.weights.h5')

In [19]:
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# إجراء التنبؤات
predictions = model.predict(X_test_padded)
predicted_classes = (predictions > 0.5).astype("int32")

# حساب الدقة
acc = accuracy_score(y_test_encoded, predicted_classes)
print(f"Accuracy: {acc:.4f}")

# تقرير الأداء (precision, recall, f1-score)
print("\nClassification Report:")
print(classification_report(y_test_encoded, predicted_classes))


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Accuracy: 0.9860

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99       866
         1.0       0.98      0.91      0.95       134

    accuracy                           0.99      1000
   macro avg       0.99      0.95      0.97      1000
weighted avg       0.99      0.99      0.99      1000



In [17]:
loss, accuracy = model.evaluate(X_test_padded, y_test_encoded)
print(f"Test Accuracy: {accuracy:.4f}")


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9892 - loss: 0.0498
Test Accuracy: 0.9860
